# **CB 2 Metode Numerik (File `.ipynb`)**

In [1]:
# setup
import numpy as np

### **Diketahui sebagai berikut:**

Untuk variabel keputusan sebagai berikut ($x$): 

$x_1, x_2, x_3, x_4, x_5, x_6$ 

Dimana variabel tersebut melambangkan total laju aliran paket di masing-masing router 1 sampai 6 dalam paket/detik.

Untuk Vektor konstanta sebagai berikut ($b$):

$$b = \begin{bmatrix} 50 \\ 20 \\ 30 \\ 10 \\ 40 \\ 60 \end{bmatrix}$$

In [ ]:
# implementasi untuk vektor konstanta b

b = np.array([50, 20, 30, 10, 40, 60]).reshape(-1, 1) # method reshape digunakan agar berbentuk vertikal

Selanjutnya adalah nilai-nilai di dalam variabel `A` diperoleh langsung dari koefisien variabel $x_1$ sampai $x_6$ setelah persamaan aliran setiap router dipindahkan ke ruas kiri (bentuk baku $Ax = b$). 

Setiap baris pada array `A` merepresentasikan satu persamaan router, dan setiap kolom berkorespondensi dengan variabel $x_1$ hingga $x_6$:

* **Baris 1 (Router 1):** Dari persamaan $x_1 = 50 + 0.2x_2 + 0.3x_3 \rightarrow 1x_1 - 0.2x_2 - 0.3x_3 = 50$, diambil koefisiennya $\rightarrow$ `[ 1.0, -0.2, -0.3, 0.0, 0.0, 0.0]`
* **Baris 2 (Router 2):** Dari persamaan $x_2 = 20 + 0.4x_1 + 0.1x_4 \rightarrow -0.4x_1 + 1x_2 - 0.1x_4 = 20$, diambil koefisiennya $\rightarrow$ `[-0.4, 1.0, 0.0, -0.1, 0.0, 0.0]`
* **Baris 3 (Router 3):** Dari persamaan $x_3 = 30 + 0.2x_1 + 0.4x_5 \rightarrow -0.2x_1 + 1x_3 - 0.4x_5 = 30$, diambil koefisiennya $\rightarrow$ `[-0.2, 0.0, 1.0, 0.0, -0.4, 0.0]`
* **Baris 4 (Router 4):** Dari persamaan $x_4 = 10 + 0.3x_2 + 0.2x_6 \rightarrow -0.3x_2 + 1x_4 - 0.2x_6 = 10$, diambil koefisiennya $\rightarrow$ `[ 0.0, -0.3, 0.0, 1.0, 0.0, -0.2]`
* **Baris 5 (Router 5):** Dari persamaan $x_5 = 40 + 0.3x_3 + 0.3x_6 \rightarrow -0.3x_3 + 1x_5 - 0.3x_6 = 40$, diambil koefisiennya $\rightarrow$ `[ 0.0, 0.0, -0.3, 0.0, 1.0, -0.3]`
* **Baris 6 (Router 6):** Dari persamaan $x_6 = 60 + 0.4x_4 + 0.2x_5 \rightarrow -0.4x_4 - 0.2x_5 + 1x_6 = 60$, diambil koefisiennya $\rightarrow$ `[ 0.0, 0.0, 0.0, -0.4, -0.2, 1.0]`

**Aturan Pembentukan:**
1. **Diagonal Utama (`1.0`)**: Bernilai positif $1$ karena menyatakan variabel router itu sendiri.
2. **Nilai Negatif (`-0.2`, `-0.3`, dll.)**: Berasal dari perpindahan posisi variabel limpahan trafik dari ruas kanan ke ruas kiri.
3. **Nilai Nol (`0.0`)**: Menandakan bahwa router tersebut tidak memiliki jalur limpahan langsung satu sama lain.

In [ ]:
# inisisasi variabel A
A = np.array([
    [ 1.0, -0.2, -0.3,  0.0,  0.0,  0.0],
    [-0.4,  1.0,  0.0, -0.1,  0.0,  0.0],
    [-0.2,  0.0,  1.0,  0.0, -0.4,  0.0],
    [ 0.0, -0.3,  0.0,  1.0,  0.0, -0.2],
    [ 0.0,  0.0, -0.3,  0.0,  1.0, -0.3],
    [ 0.0,  0.0,  0.0, -0.4, -0.2,  1.0]
])

Selanjutnya ini ada kode function untuk menghitung dengan metode gauss jordan:

Yang dibutuhkan hanya variabel matriks `A` dan variabel vektor `b`.

Cara kerja (secara bertahap setiap baris atau algoritma deskriptif):

1. Function menerima parameter matriks `A` dan vektor `b` dan juga konversi variabel A dan B ke float
2. Inisiasi variabel `n` yang bernilai panjang dari `b` (jumlah baris atau variabel sistem persamaan).
3. Inisiasi variabel `M` sebagai matriks augmented yang nanti akan dimasukan.
4. Melakukan looping untuk menggabungkan setiap baris matriks `A` dengan elemen vektor `b` di kolom terakhirnya, lalu memasukkannya ke dalam matriks augmented `M`.
5. Melakukan looping menggunakan indeks `k` untuk menelusuri elemen diagonal utama, kemudian membagi seluruh elemen pada baris `k` dengan nilai pivot tersebut agar elemen diagonal utamanya bernilai $1$.
6. Melakukan looping untuk memeriksa setiap baris `i` (di mana `i` tidak sama dengan `k`), lalu mengurangkan baris tersebut dengan kelipatan baris pivot untuk mengenolkan seluruh elemen di kolom `k` (baik di atas maupun di bawah diagonal utama).
7. Mengambil nilai-nilai hasil akhir di kolom terakhir dari matriks `M` yang telah tereduksi menjadi matriks identitas, lalu mengembalikannya sebagai vektor solusi eksak $x_1$ hingga $x_6$.

In [6]:
# implementasi kode hasil dari 

def gauss_jordan(A, b):
    # konversi integer ke float
    A_float = np.array(A, dtype=float)
    b_float = np.array(b, dtype=float).flatten()
    
    n = len(b)
    # buat matriks augmented [A | b]
    M = []
    for i in range(n):
        row = list(A_float[i]) + [b_float[i]]
        M.append(row)
        
    for k in range(n):
        # normalisasi baris pivot agar elemen diagonal utama jadi 1
        pivot = M[k][k]
        for j in range(n + 1):
            M[k][j] /= pivot
            
        # eliminasi elemen di atas dan di bawah pivot
        for i in range(n):
            if i != k:
                faktor = M[i][k]
                for j in range(n + 1):
                    M[i][j] -= faktor * M[k][j]
                    
    # ambil hasil solusi di kolom terakhir
    solusi = [M[i][n] for i in range(n)]
    return solusi

In [7]:
# memasukan nilai dan hasil dari perhitungan tersebut
gauss_jordan(A,b)

[np.float64(87.62823062926755),
 np.float64(59.806178547904295),
 np.float64(85.55664973228902),
 np.float64(47.548862961972645),
 np.float64(95.07750901608875),
 np.float64(98.03504698800681)]

In [ ]:
def gauss_seidel(A, b, x0=None, es=0.0001, max_iter=1000):
    n = len(b)
    b_list = np.array(b).flatten()
    if x0 is not None:
        x = list(x0)
    else:
        x = [0.0] * n
        
    for iterasi in range(1, max_iter + 1):
        x_lama = x[:]
        for i in range(n):
            sigma = 0
            for j in range(n):
                if j != i:
                    sigma += A[i][j] * x[j]
            x[i] = (b_list[i] - sigma) / A[i][i]
            
        galat_maks = 0.0
        for i in range(n):
            if x[i] != 0:
                galat_relatif = abs((x[i] - x_lama[i]) / x[i])
                galat_maks = max(galat_maks, galat_relatif)
            
        if galat_maks < es:
            break
            
    return x